# Project Visual environment / Entorno visual del proyecto / Entorn Visual del projecte

<!-- en: --><b> This notebook allows running the `etl.ipynb`, `modelitza.ipynb` and `predict.ipynb` notebooks interactively.<br>Unlike the original (which is programmed in Textual), this one's mostly created using <i>vibe coding</i> techniques.</b><br><br>
<!-- es: --><i> Este notebook permite lanzar los notebooks `etl.ipynb`, `modelitza.ipynb` y `predict.ipynb` de manera interactiva.<br>A diferencia del original (que está programado en Textual), este está realizado en su mayoría utilizando técnicas de <b>vibe coding</b>.</i><br><br>
<!-- ca: --><i> Aquest notebook permet llançar els notebooks `etl.ipynb`, `modelitza.ipynb` i `predict.ipynb` de manera interactiva.<br>A diferència de l'original (que està programat en Textual), aquest està realitaz en majoria utilitzant tècniques de <b>vibe coding</b>.</i></br><br>



In [ ]:
import ipywidgets as widgets
import subprocess
import threading
import time
import sys
import os
try:
    from ipyfilechooser import FileChooser
except ImportError:
    class FileChooser:
        def __init__(self):
            self.title = 'Select CSV file'
            self.show_only_dirs = False
            self.selected = None
            self._widget = widgets.Text(description='Path CSV:')
        @property
        def as_widget(self):
            return self._widget
        def _repr_mimebundle_(self, **kwargs):
            return self._widget._repr_mimebundle_(**kwargs)
        @property
        def selected(self):
            return self._widget.value
        def reset(self):
            self._widget.value = ''


# ================= TRANSLATIONS / TRADUCCIONES / TRADUCCIONS =================

locales = {
    'en': {
        'panel_title': "<h2 style='color:#2c3e50;'>🚀 Project Panel</h2>",
        'lang_desc': "Language:",
        'exec_desc': "Run:",
        'basic_chk': "--basic (Perform basic analysis)",
        'mode_desc': "Mode:",
        'mode_none': "None",
        'mode_rolling': "--rolling (Retrain all data)",
        'mode_train': "--train (New training data)",
        'fc_title': "Select CSV file for --predir (optional)",
        'reset_btn': "⏹️ Reset file selection",
        'run_btn': "Run",
        'abort_btn': "Abort",
        'time_elapsed': "⏱️ Elapsed time:",
        'time_processing': "⏱️ Processing:",
        'time_finished': "⏱️ FINISHED in",
        'log_waiting': "Waiting for execution...",
        'log_title': "<hr><b>📋 Active Processing Console:</b>",
        
        # Mensajes de salida interna
        'log_req_ok': "Button request processed.",
        'log_dir': "> Active directory:",
        'log_cmd': "> Running notebook:",
        'err_crit': "\n[CRITICAL ERROR] subprocess.Popen finished instantly:",
        'err_iter': "\n[!] Exception processing iterator:",
        'err_wait': "\n[!] Exception processing wait:",
        'msg_ok': "✅ Process finished successfully.",
        'msg_abort': "🛑 Process aborted or cancelled.",
        'msg_err': "❌ Process finished asynchronously with error (code",
        'msg_kill': "\n[!] Termination request sent... killing subprocess.",
        'test_exec': "import time, sys; print('- Starting subprocess test phase...'); time.sleep(1); print('++ STDOUT correctly captured'); sys.stderr.write('++ STDERR correctly captured\\n'); time.sleep(1); print('- Finished')"
    },
    'es': {
        'panel_title': "<h2 style='color:#2c3e50;'>🚀 Panel del Proyecto</h2>",
        'lang_desc': "Idioma:",
        'exec_desc': "Ejecutar:",
        'basic_chk': "--basic (Realizar un análisis básico)",
        'mode_desc': "Modo:",
        'mode_none': "Ninguno",
        'mode_rolling': "--rolling (Reentrenar todo)",
        'mode_train': "--train (Nuevos datos)",
        'fc_title': "Seleccionar fichero CSV para --predir (opcional)",
        'reset_btn': "⏹️ Restablecer fichero",
        'run_btn': "Ejecutar",
        'abort_btn': "Abortar",
        'time_elapsed': "⏱️ Tiempo transcurrido:",
        'time_processing': "⏱️ Procesando:",
        'time_finished': "⏱️ FINALIZADO en",
        'log_waiting': "Esperando ejecución...",
        'log_title': "<hr><b>📋 Consola de Procesamiento Activo:</b>",
        
        'log_req_ok': "Petición de botón procesada.",
        'log_dir': "> Directorio activo:",
        'log_cmd': "> Ejecutando notebook:",
        'err_crit': "\n[ERROR CRÍTICO] subprocess.Popen finalizó instantáneamente:",
        'err_iter': "\n[!] Excepción procesando iterador iter:",
        'err_wait': "\n[!] Excepción procesando wait:",
        'msg_ok': "✅ Proceso finalizado correctamente.",
        'msg_abort': "🛑 Proceso abortado o cancelado.",
        'msg_err': "❌ Proceso finalizó asíncronamente con error (código",
        'msg_kill': "\n[!] Solicitud de Terminación enviada... matando al subproceso.",
        'test_exec': "import time, sys; print('- Iniciando fase test de subproceso...'); time.sleep(1); print('++ STDOUT capturado correctamente'); sys.stderr.write('++ STDERR capturado correctamente\\n'); time.sleep(1); print('- Terminado')"
    },
    'ca': {
        'panel_title': "<h2 style='color:#2c3e50;'>🚀 Tauler del Projecte</h2>",
        'lang_desc': "Idioma:",
        'exec_desc': "Executar:",
        'basic_chk': "--basic (Fer un anàlisi bàsic)",
        'mode_desc': "Mode:",
        'mode_none': "Cap",
        'mode_rolling': "--rolling (Reentrenar tot)",
        'mode_train': "--train (Noves dades)",
        'fc_title': "Seleccionar fitxer CSV per a --predir (opcional)",
        'reset_btn': "⏹️ Restablir fitxer",
        'run_btn': "Executar",
        'abort_btn': "Avortar",
        'time_elapsed': "⏱️ Temps transcorregut:",
        'time_processing': "⏱️ Processant:",
        'time_finished': "⏱️ FINALITZAT en",
        'log_waiting': "Esperant execució...",
        'log_title': "<hr><b>📋 Consola de Processament Actiu:</b>",
        
        'log_req_ok': "Petició de botó processada.",
        'log_dir': "> Directori actiu:",
        'log_cmd': "> Executant notebook:",
        'err_crit': "\n[ERROR CRÍTIC] subprocess.Popen ha finalitzat instantàniament:",
        'err_iter': "\n[!] Excepció processant l'iterador:",
        'err_wait': "\n[!] Excepció processant wait:",
        'msg_ok': "✅ Procés finalitzat correctament.",
        'msg_abort': "🛑 Procés avortat o cancel·lat.",
        'msg_err': "❌ Procés finalitzat asíncronament amb error (codi",
        'msg_kill': "\n[!] Sol·licitud de Terminació enviada... matant al subprocés.",
        'test_exec': "import time, sys; print('- Iniciant fase test de subprocés...'); time.sleep(1); print('++ STDOUT capturat correctament'); sys.stderr.write('++ STDERR capturat correctament\\n'); time.sleep(1); print('- Acabat')"
    }
}

current_lang = 'en'

# en: Basic styles
# es: Estilos básicos
# ca: Estils bàsics

style = {'description_width': 'initial'}
layout = {'width': 'max-content'}

# en: Global state
# es: Estado global
# ca: Estat global

is_running = False
current_process = None

# ---- Widgets HTML ----

html_panel_title = widgets.HTML(value=locales[current_lang]['panel_title'])
html_log_title = widgets.HTML(value=locales[current_lang]['log_title'])
html_br = widgets.HTML("<br>")

# en: ---- Language and Selection Widgets ----
# es: ---- Widgets de Idioma y Selección ----
# ca: ---- Widgets d'Idioma i Selecció ----

lang_dropdown = widgets.Dropdown(
    options=[('English', 'en'), ('Español', 'es'), ('Català', 'ca')],
    value='en',
    description=locales[current_lang]['lang_desc'],
    style=style,
    layout=layout
)

opts_notebook = ['etl.ipynb', 'modelitza.ipynb', 'predict.ipynb']
notebook_dropdown = widgets.Dropdown(
    options=opts_notebook,
    value='etl.ipynb',
    description=locales[current_lang]['exec_desc'],
    style=style,
    layout=layout
)

# en: ---- Specific Options ----
# es: ---- Opciones Específicas ----
# ca: ---- Opcions Específiques ----

basic_checkbox = widgets.Checkbox(
    value=False,
    description=locales[current_lang]['basic_chk'],
    style=style
)
modelitza_options = widgets.VBox([basic_checkbox])

opts_predict = [ 
    (locales[current_lang]['mode_none'], ''), 
    (locales[current_lang]['mode_rolling'], '--rolling'), 
    (locales[current_lang]['mode_train'], '--train')
]
predict_mode = widgets.RadioButtons(
    options=opts_predict,
    value='',
    description=locales[current_lang]['mode_desc'],
    style=style
)

file_chooser = FileChooser()
if hasattr(file_chooser, 'title'):
    file_chooser.title = locales[current_lang]['fc_title']

reset_fc_button = widgets.Button(
    description=locales[current_lang]['reset_btn'], 
    button_style='warning', 
    icon='eraser',
    layout={'width': 'max-content','display': 'none'}
)
def rst_filechooser_clk(b):
    if hasattr(file_chooser, 'reset'):
        file_chooser.reset()
        b.layout.display = 'none'
        if hasattr(file_chooser, 'title'):
            file_chooser.title = locales[current_lang]['fc_title']
reset_fc_button.on_click(rst_filechooser_clk)

fc_widget = file_chooser.as_widget if hasattr(file_chooser, 'as_widget') else file_chooser
predict_options = widgets.VBox([predict_mode, fc_widget, reset_fc_button])

def si_fich_escollit(canvi):
    if canvi.selected:
        reset_fc_button.layout.display = ''

file_chooser.register_callback(si_fich_escollit)


# en: Dynamic container
# es: Contenedor dinámico
# ca: Contenidor dinàmic

options_container = widgets.VBox()

def update_options(*args):
    notebook = notebook_dropdown.value
    if notebook == 'modelitza.ipynb':
        options_container.children = [modelitza_options]
    elif notebook == 'predict.ipynb':
        options_container.children = [predict_options]
    else:
        options_container.children = []

notebook_dropdown.observe(update_options, 'value')
update_options()

# en: ---- Execution Controls ----
# es: ---- Controles de Ejecución ----
# ca: ---- Controls d'execució ----

run_button = widgets.Button(description=locales[current_lang]['run_btn'], button_style='success', icon='play')
abort_button = widgets.Button(description=locales[current_lang]['abort_btn'], button_style='danger', disabled=True, icon='stop')
time_label = widgets.Label(value=f"{locales[current_lang]['time_elapsed']} 0s")

controls_box = widgets.HBox([run_button, abort_button, time_label])

# en: ---- Output ----
# es: ---- Salida ----
# ca: ---- Sortida ----

output_log = widgets.Textarea(
    value='',
    placeholder=locales[current_lang]['log_waiting'],
    layout={'width': '100%', 'height': '400px'}
)

def log_msg(msg):
    output_log.value += str(msg) + "\n"

# en: ---- Language Logic ----
# es: ---- Lógica de Idioma ----
# ca: ---- Lògica d'Idioma ----

def update_language(change):
    global current_lang
    new_lang = change.new
    t = locales[new_lang]
    current_lang = new_lang
    
    # en: Update HTML titles
    # es: Actualizar títulos HTML
    # ca: Actualitzar títols HTML

    html_panel_title.value = t['panel_title']
    html_log_title.value = t['log_title']
    
    # en: Update Dropdowns and Labels
    # es: Actualizar Dropdowns y Labels
    # ca: Actualitzar Dropdowns i Labels

    lang_dropdown.description = t['lang_desc']
    notebook_dropdown.description = t['exec_desc']
        
    # en: Update Checkboxes and Radios
    # es: Actualizar Checkboxes y Radios
    # ca: Actualitzar Checkboxes i Radios

    basic_checkbox.description = t['basic_chk']
    predict_mode.description = t['mode_desc']
    
    old_pred_val = predict_mode.value
    predict_mode.options = [ 
        (t['mode_none'], ''), 
        (t['mode_rolling'], '--rolling'), 
        (t['mode_train'], '--train')
    ]
    predict_mode.value = old_pred_val
    
    if hasattr(file_chooser, 'title'):
        file_chooser.title = t['fc_title']
        
    reset_fc_button.description = t['reset_btn']
        
    # en: Update Buttons
    # es: Actualizar Botones
    # ca: Actualitzar Botons

    run_button.description = t['run_btn']
    abort_button.description = t['abort_btn']
    output_log.placeholder = t['log_waiting']
    
    # en: If not running, update the wait time to zero
    # es: Si no está ejecutando, actualizar el tiempo de espera a cero
    # ca: Si no està executant, actualitzar el temps de espera a zero

    if not is_running and "0s" in time_label.value:
        time_label.value = f"{t['time_elapsed']} 0s"

lang_dropdown.observe(update_language, 'value')

# en: ---- Functionality ----
# es: ---- Funcionalidad ----
# ca: ---- Funcionalitat ----

def run_notebook(b):
    global is_running, current_process
    t = locales[current_lang]
    if is_running:
        return
        
    is_running = True
    run_button.disabled = True
    abort_button.disabled = False
    notebook_dropdown.disabled = True
    lang_dropdown.disabled = True
    output_log.value = "" 
    
    cwd_path = os.path.abspath('')
    notebook_name = notebook_dropdown.value
    
    args = []
    if notebook_name == 'modelitza.ipynb':
        if basic_checkbox.value:
            args.append("--basic")
    elif notebook_name == 'predict.ipynb':
        if predict_mode.value:
            args.append(predict_mode.value)
        if file_chooser.selected:
            args.extend(["--predir", file_chooser.selected])
            
    cmd = [sys.executable, '-u', '-m', 'IPython', notebook_name]
    if args:
        cmd.append("--")
        cmd.extend(args)
            
    log_msg(f"[{time.strftime('%H:%M:%S')}] {t['log_req_ok']}")
    log_msg(f"{t['log_dir']} {cwd_path}")
    log_msg(f"{t['log_cmd']} {notebook_name}")
    log_msg("-" * 40)
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    
    try:
        current_process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
            cwd=cwd_path,
            encoding='utf-8',
            errors='replace'
        )
    except Exception as e:
        log_msg(f"{t['err_crit']} {e}")
        is_running = False
        run_button.disabled = False
        abort_button.disabled = True
        notebook_dropdown.disabled = False
        lang_dropdown.disabled = False
        return
    
    start_time = time.time()
    
    def output_reader():
        global is_running, current_process
        try:
            for line in iter(current_process.stdout.readline, ''):
                if line:
                    log_msg(line.rstrip())
        except Exception as e:
             log_msg(f"{t['err_iter']} {e}")
                
        try:
            current_process.wait()
        except Exception as e:
            log_msg(f"{t['err_wait']} {e}")
        
        log_msg("\n" + "-" * 40)
        if current_process.returncode == 0:
            log_msg(f"[{time.strftime('%H:%M:%S')}] {t['msg_ok']}")
        elif current_process.returncode < 0:
            log_msg(f"[{time.strftime('%H:%M:%S')}] {t['msg_abort']}")
        else:
            log_msg(f"[{time.strftime('%H:%M:%S')}] {t['msg_err']} {current_process.returncode}).")
            
        is_running = False
        run_button.disabled = False
        abort_button.disabled = True
        notebook_dropdown.disabled = False
        lang_dropdown.disabled = False
        time_label.value = f"{t['time_finished']} {int(time.time() - start_time)}s"
        
    reader_thread = threading.Thread(target=output_reader)
    reader_thread.daemon = True
    reader_thread.start()
    
    def time_updater():
        while is_running:
            elapsed = time.time() - start_time
            time_label.value = f"{t['time_processing']} {int(elapsed)}s"
            time.sleep(1)
            
    time_thread = threading.Thread(target=time_updater)
    time_thread.daemon = True
    time_thread.start()

def abort_notebook(b):
    global is_running, current_process
    t = locales[current_lang]
    if current_process and is_running:
        log_msg(f"{t['msg_kill']}")
        current_process.kill()
        
run_button.on_click(run_notebook)
abort_button.on_click(abort_notebook)

app_layout = widgets.VBox([
    html_panel_title,
    lang_dropdown,
    html_br,
    notebook_dropdown,
    options_container,
    html_br,
    controls_box,
    html_log_title,
    output_log
])

display(app_layout)
